# XAI — LRP attributions (CESM2-LE CNN)

Loads pre-trained CNN models and their TVT splits, then computes and saves LRP-z attributions for each split × seed.

> **Must run in a fresh kernel** — `src.xai.lrp` calls `tf.compat.v1.disable_eager_execution()` at import time.  
> Do **not** import or run this notebook in the same kernel session as `cnn_train.ipynb`.

**Prerequisites:** run `scripts/04_cesm2le_cnn_train.py` (or `cnn_train.ipynb`) to train the models first.

In [1]:
import sys
from pathlib import Path
import numpy as np
import netCDF4 as nc

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths
from src.cnn.splits import load_tvt_split
from src.cnn.train  import load_model
from src.xai.lrp    import strip_sigmoid, compute_lrp_z, save_lrp

2026-03-31 19:18:07.657066: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-31 19:18:07.659172: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-31 19:18:07.688913: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-31 19:18:07.688943: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-31 19:18:07.688974: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

## Config
Match `N_SPLITS` and `N_SEEDS` to what was used during training.

In [2]:
N_SPLITS   = 3     # number of TVT splits that were trained
N_SEEDS    = 3     # number of seeds per split
CHUNK_SIZE = 100   # samples per LRP batch — reduce if you hit memory errors

SKIP_EXISTING = True   # set False to recompute even if the output file exists

## Load lat / lon from CESM2-LE grid file

In [3]:
ds_grid = nc.Dataset(paths.CESM2LE_GRID_FILE, 'r')
lat_1d  = np.array(ds_grid.variables['lat'][:])   # (nlat,)
lon_1d  = np.array(ds_grid.variables['lon'][:])   # (nlon,)
ds_grid.close()
print(f'Grid: lat {lat_1d.shape}, lon {lon_1d.shape}')

Grid: lat (192,), lon (288,)


## Compute and save LRP for all splits × seeds

In [4]:
for split_idx in range(N_SPLITS):
    print(f'\n{"─"*60}')
    print(f'Split {split_idx}')
    print(f'{"─"*60}')

    split_path = paths.tvt_split_path(split_idx)
    if not split_path.exists():
        raise FileNotFoundError(
            f'TVT split not found: {split_path}\n'
            f'Run scripts/03_cesm2le_tvt_splits.py first.'
        )
    split = load_tvt_split(split_path)
    x_tr  = split['sst_tr'][:, :, :, np.newaxis]   # (n_tr, nx, ny, 1)
    print(f'  Training data: {x_tr.shape}')

    for run_idx in range(N_SEEDS):
        out_path = paths.attribution_path(split_idx, run_idx)

        if SKIP_EXISTING and out_path.exists():
            print(f'  run {run_idx} — already exists, skipping.')
            continue

        model_p = paths.model_path(split_idx, run_idx)
        if not model_p.exists():
            print(f'  run {run_idx} — model not found ({model_p}), skipping.')
            continue

        print(f'  run {run_idx}: loading model ...', end=' ', flush=True)
        model        = load_model(paths.MODELS_DIR, split_idx, run_idx)
        model_logits = strip_sigmoid(model)
        print('done')

        print(f'  run {run_idx}: computing LRP (chunk_size={CHUNK_SIZE}) ...')
        attributions = compute_lrp_z(model_logits, x_tr, chunk_size=CHUNK_SIZE)
        # shape: (n_chunks, CHUNK_SIZE, nx, ny, 1)

        save_lrp(
            attributions, lat_1d, lon_1d,
            savepath=out_path,
            split_idx=split_idx,
            run_idx=run_idx,
        )

print('\nAll LRP attributions complete.')


────────────────────────────────────────────────────────────
Split 0
────────────────────────────────────────────────────────────
  Training data: (4080, 192, 288, 1)
  run 0: loading model ... 

2026-03-31 19:18:22.757192: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2026-03-31 19:18:22.802518: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-03-31 19:18:22.815918: W tensorflow/c/c_api.cc:305] Operation '{name:'dense/kernel/Assign' id:78 op device:{requested: '', assigned: ''} def:{{{node dense/kernel/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](dense/kernel, dense/kernel/Initializer/stateless_random_uniform)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the f

done
  run 0: computing LRP (chunk_size=100) ...


2026-03-31 19:18:24.025462: W tensorflow/c/c_api.cc:305] Operation '{name:'bias_2/Assign' id:411 op device:{requested: '', assigned: ''} def:{{{node bias_2/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](bias_2, bias_2/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-03-31 19:18:24.050804: W tensorflow/c/c_api.cc:305] Operation '{name:'gradients/MaxNeuronSelection/Max_grad/range' id:421 op device:{requested: '', assigned: ''} def:{{{node gradients/MaxNeuronSelection/Max_grad/range}} = Range[Tidx=DT_INT32, _class=["loc:@MaxNeuronSelection/Max"], _has_manual_control_dependencies=true](gradients/MaxNeuronSelection/Max_grad/range/start, gradients/MaxNeuronSelection/Max_grad/Size, gradients/MaxNeuronSelection/Max_grad/range/delta)}}' was changed by s

  LRP chunk 10/40 done
  LRP chunk 20/40 done
  LRP chunk 30/40 done
  LRP chunk 40/40 done
  LRP attributions saved → /mnt/tank/Oceanography/data/OGCM/LLC/Fronts/lohoff/arcticWatch/results/attributions/lrp_jja_split0_run0.nc
  run 1: loading model ... 

...

  LRP chunk 10/40 done
  LRP chunk 20/40 done
  LRP chunk 30/40 done
  LRP chunk 40/40 done
  LRP attributions saved → /mnt/tank/Oceanography/data/OGCM/LLC/Fronts/lohoff/arcticWatch/results/attributions/lrp_jja_split2_run1.nc
  run 2: loading model ... 

2026-03-31 19:23:07.842992: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_8/bias/Assign' id:4259 op device:{requested: '', assigned: ''} def:{{{node dense_8/bias/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](dense_8/bias, dense_8/bias/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-03-31 19:23:08.124023: W tensorflow/c/c_api.cc:305] Operation '{name:'total_8/Assign' id:4304 op device:{requested: '', assigned: ''} def:{{{node total_8/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](total_8, total_8/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify node

done
  run 2: computing LRP (chunk_size=100) ...


2026-03-31 19:23:09.372413: W tensorflow/c/c_api.cc:305] Operation '{name:'kernel_24/Assign' id:4544 op device:{requested: '', assigned: ''} def:{{{node kernel_24/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](kernel_24, kernel_24/Initializer/stateless_random_uniform)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2026-03-31 19:23:09.495636: W tensorflow/c/c_api.cc:305] Operation '{name:'gradients_64/MaxNeuronSelection_8/Max_grad/range' id:4597 op device:{requested: '', assigned: ''} def:{{{node gradients_64/MaxNeuronSelection_8/Max_grad/range}} = Range[Tidx=DT_INT32, _class=["loc:@MaxNeuronSelection_8/Max"], _has_manual_control_dependencies=true](gradients_64/MaxNeuronSelection_8/Max_grad/range/start, gradients_64/MaxNeuronSelection_8/Max_grad/Size, gradients_64/Ma

  LRP chunk 10/40 done
  LRP chunk 20/40 done
  LRP chunk 30/40 done
  LRP chunk 40/40 done
  LRP attributions saved → /mnt/tank/Oceanography/data/OGCM/LLC/Fronts/lohoff/arcticWatch/results/attributions/lrp_jja_split2_run2.nc

All LRP attributions complete.
